---

# Playground S6E4: Predicting Irrigation Need 💧
*Agricultural water management and multi-class tabular classification*

---

---

# Introduction 

---

**Playground Series S6E4** is a multi-class tabular classification task: predict agricultural irrigation needs (Low, Medium, High) based on synthetic climate, soil, and crop condition data. Performance is measured by Balanced Accuracy, making the accurate prediction of the severely underrepresented minority classes just as critical as the majority class. Participants must submit predicted class labels in a standard CSV format, with a strong emphasis on robust cross-validation to prevent overfitting.

---

# Initialization 

---

---

## Environment

In [1]:
import os, platform, sys
print('Python :', sys.version)
print('OS     :', platform.system(), platform.release())
print('CWD    :', os.getcwd())

Python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
OS     : Linux 6.6.113+
CWD    : /kaggle/working


---

# Libraries

In [2]:
# 1. Data Manipulation & Array Operations
import pandas as pd
import numpy as np

# 2. Preprocessing & Cross-Validation
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# 3. Hyperparameter Tuning
import optuna

# 4. Machine Learning Algorithms
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# 5. System & Warning Handling (Optional but recommended)
import warnings
warnings.filterwarnings('ignore')

---

# Data Loading and Preprocessing

---

---

## Loading Dataset 

Kita mendefinisikan lokasi folder dataset yang pasti (BASE_PATH), lalu langsung menyuruh Pandas untuk membaca ketiga file CSV (train, test, dan sample_submission) ke dalam memori. Di akhir, kita mencetak dimensi datanya untuk memastikan baris dan kolom sudah terbaca dengan benar.

In [3]:
BASE_PATH = '/kaggle/input/competitions/playground-series-s6e4'

train_df = pd.read_csv(f'{BASE_PATH}/train.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test.csv')
sample_sub_df = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

Train data shape: (630000, 21)
Test data shape: (270000, 20)


---

## Separating Fetures and Target

In this cell, we define the exact dataset folder location (**BASE_PATH**) and instruct **Pandas** to load the three primary CSV files (`train.csv`, `test.csv`, and `sample_submission.csv`) into memory. Finally, we display the data dimensions (shape) to ensure that all rows and columns have been imported correctly.

In [4]:
X = train_df.drop(columns=['id', 'Irrigation_Need'])
y_raw = train_df['Irrigation_Need']
X_test = test_df.drop(columns=['id'])

---

## Target Encoding

Modeling algorithms only understand numbers. Therefore, we map the target levels, which are originally in text format (`Low`, `Medium`, `High`), into numerical variables (0, 1, 2) that are ready to be fed into the Gradient Boosting algorithm. The transformed results are stored in the variable `y`.

In [5]:
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y = y_raw.map(target_mapping)

---

## Categorical Feature Encoding

This section is responsible for handling the remaining columns that are in text format (such as crop type or land status). 

* `select_dtypes` automatically identifies which columns contain text data.
* Inside the loop, we temporarily combine the columns from the training and test data (`combined_data`). This is a crucial trick to ensure that the `LabelEncoder` can recognize every categorical variation present across both datasets.
* Once the encoder learns the entire vocabulary, the text in both the train and test sets is permanently converted into numerical codes.
* The `print` statement at the end serves as a final sanity check to verify that the number of rows, columns, and the target class distribution (class imbalance) are accurate.

In [6]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    
    combined_data = pd.concat([X[col], X_test[col]]).astype(str)
    le.fit(combined_data)
    
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print(f"Final Features (X) shape: {X.shape}")
print(f"Final Test Features (X_test) shape: {X_test.shape}")
print(f"Target (y) value counts:\n{y.value_counts()}")

Final Features (X) shape: (630000, 19)
Final Test Features (X_test) shape: (270000, 19)
Target (y) value counts:
Irrigation_Need
0    369917
1    239074
2     21009
Name: count, dtype: int64


---

# Feature Engineering

---

---
## Defining the Feature Engineering Function

This cell defines a reusable function named `create_features`. Instead of relying only on raw data, this function creates new columns (interaction features) based on logical agricultural relationships—such as dividing temperature by soil moisture to calculate a "Dryness Stress Index." We use `if` statements to safely check if the required columns exist before performing the mathematical operations, preventing the code from crashing.

In [7]:
def create_features(df):
    df_engineered = df.copy()
    
    # 1. Dryness Stress Index
    if 'Temperature_C' in df_engineered.columns and 'Soil_Moisture' in df_engineered.columns:
        df_engineered['Dryness_Stress_Index'] = df_engineered['Temperature_C'] / (df_engineered['Soil_Moisture'] + 1e-5)
        
    # 2. Climate Severity
    if 'Temperature_C' in df_engineered.columns and 'Precipitation_mm' in df_engineered.columns:
        df_engineered['Temp_Minus_Precip'] = df_engineered['Temperature_C'] - df_engineered['Precipitation_mm']
        
    # 3. Water Retention Potential
    if 'Soil_Moisture' in df_engineered.columns and 'Precipitation_mm' in df_engineered.columns:
        df_engineered['Moisture_Precip_Interaction'] = df_engineered['Soil_Moisture'] * df_engineered['Precipitation_mm']
        
    # 4. Crop & Mulch Interaction
    if 'Crop_Growth_Stage' in df_engineered.columns and 'Mulching_Used' in df_engineered.columns:
        df_engineered['Crop_Mulch_Interaction'] = (df_engineered['Crop_Growth_Stage'].astype(int) * 10) + df_engineered['Mulching_Used'].astype(int)
        
    return df_engineered

---
## Applying the Function to the Datasets

Here, we execute the `create_features` function we just built. We apply it to both our training features (`X`) and our test features (`X_test`). It is critical to apply the exact same feature engineering pipeline to both datasets so that the machine learning model receives consistent data structures when training and predicting.

In [8]:
X = create_features(X)
X_test = create_features(X_test)

---
## Verifying the Newly Created Features

This final step acts as a verification checkpoint. We print the new shape (dimensions) of our datasets to confirm that the new columns were added successfully. By subtracting the original column names from the current column names, we dynamically extract and print a list of the specific engineered features that were actually created. This is highly useful for debugging in case a column name was misspelled in the dataset.

In [9]:
print(f"Engineered Features (X) shape: {X.shape}")
print(f"Engineered Test Features (X_test) shape: {X_test.shape}")

original_cols = set(train_df.columns) - {'id', 'Irrigation_Need'}
new_cols = list(set(X.columns) - original_cols)

print(f"\nNew features successfully created: {new_cols}")

Engineered Features (X) shape: (630000, 21)
Engineered Test Features (X_test) shape: (270000, 21)

New features successfully created: ['Crop_Mulch_Interaction', 'Dryness_Stress_Index']


---
# Cross-Validation Strategy (Stratified K-Fold)
---

---
## Initializing Stratified K-Fold

In this cell, we define the configuration for our cross-validation strategy. We use `StratifiedKFold` with 5 splits. This is extremely important for imbalanced datasets because "stratified" ensures that every single fold will have the exact same percentage of "Low", "Medium", and "High" target classes as the original training data. Setting a `RANDOM_STATE` ensures that our results are reproducible every time we run the notebook.

In [10]:
N_SPLITS = 5
RANDOM_STATE = 42

print(f"Setting up Stratified {N_SPLITS}-Fold Cross Validation...")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

Setting up Stratified 5-Fold Cross Validation...


---
## Preparing Prediction Arrays for the Ensemble

Here, we pre-allocate empty numpy arrays filled with zeros to store the predicted probabilities from our models. 
* The **OOF (Out-Of-Fold)** arrays will store the predictions made on the validation folds during training. This allows us to evaluate the true performance of the models on the entire dataset without data leakage.
* The **test_preds** arrays will accumulate the predictions made on the unseen test data.
We prepare these arrays with 3 columns because our target has 3 distinct classes, and we will output the probability for each class. We also prepare slots for three different models (LightGBM, XGBoost, and CatBoost) to be used later in our ensembling (blending) phase.

In [11]:
oof_preds_lgb = np.zeros((len(X), 3))
test_preds_lgb = np.zeros((len(X_test), 3))

oof_preds_xgb = np.zeros((len(X), 3))
test_preds_xgb = np.zeros((len(X_test), 3))

oof_preds_cat = np.zeros((len(X), 3))
test_preds_cat = np.zeros((len(X_test), 3))

print(f"Cross-validation strategy initialized with {N_SPLITS} folds.")
print(f"OOF arrays prepared for LightGBM, XGBoost, and CatBoost (Shape: {oof_preds_lgb.shape}).")

Cross-validation strategy initialized with 5 folds.
OOF arrays prepared for LightGBM, XGBoost, and CatBoost (Shape: (630000, 3)).


---
# Hyperparameter Tuning with Optuna
---

---
## Defining the Optuna Objective Function

This cell defines the `objective` function required by Optuna. 
* Inside this function, we define the **search space** (using `trial.suggest_...`), which tells Optuna the minimum and maximum boundaries for parameters like `learning_rate` or `max_depth` that it is allowed to experiment with.
* We set up a smaller, 3-fold `StratifiedKFold` specifically for the tuning phase to make the evaluation process faster. 
* For every combination of parameters Optuna guesses, it trains a LightGBM model, evaluates it using the `balanced_accuracy_score`, and returns the average score. Optuna uses this score to intelligently guess better parameters in the next round.

In [12]:
def objective(trial):
    # Define the hyperparameter search space for LightGBM
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'class_weight': 'balanced', # Crucial for imbalanced data
        'random_state': RANDOM_STATE,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
    }

    # Setup a faster 3-Fold Stratified CV inside the objective function
    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for train_idx, val_idx in skf_tune.split(X, y):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**param)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        preds = model.predict(X_val_fold)
        score = balanced_accuracy_score(y_val_fold, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)

---
## Running the Optimization Study

Here, we initialize the actual Optuna study. 
* We set the `direction='maximize'` because we want to achieve the highest possible balanced accuracy score. 
* The `study.optimize()` function triggers the execution, testing 30 different parameter combinations (`n_trials=30`). 
* Once the loop is finished, we extract the absolute best parameter combination (`study.best_params`), inject the fixed mandatory parameters back into the dictionary, and print them out. This `best_lgb_params` dictionary will be fed directly into our final model training in the next step.

In [13]:
# Create a study object and optimize for maximum balanced accuracy
study = optuna.create_study(direction='maximize', study_name="LGBM_Tuning")
study.optimize(objective, n_trials=30, show_progress_bar=True)

# Extract and save the best parameters
best_lgb_params = study.best_params

# Re-add the essential fixed parameters to the dictionary
best_lgb_params['objective'] = 'multiclass'
best_lgb_params['num_class'] = 3
best_lgb_params['class_weight'] = 'balanced'
best_lgb_params['random_state'] = RANDOM_STATE

print("\nBest LightGBM Parameters found by Optuna:")
for key, value in best_lgb_params.items():
    print(f"  {key}: {value}")
    
print(f"\nBest CV Balanced Accuracy: {study.best_value:.4f}")

[I 2026-04-28 15:02:49,014] A new study created in memory with name: LGBM_Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-28 15:09:09,588] Trial 0 finished with value: 0.966714894722116 and parameters: {'n_estimators': 877, 'learning_rate': 0.011807658302857517, 'max_depth': 7, 'num_leaves': 119, 'min_child_samples': 18, 'subsample': 0.9974569010384587, 'colsample_bytree': 0.8994218031825811, 'reg_alpha': 6.209584243226381e-06, 'reg_lambda': 0.0011689137524292783}. Best is trial 0 with value: 0.966714894722116.
[I 2026-04-28 15:11:23,956] Trial 1 finished with value: 0.9660364375940952 and parameters: {'n_estimators': 605, 'learning_rate': 0.24558497650396743, 'max_depth': 9, 'num_leaves': 147, 'min_child_samples': 42, 'subsample': 0.9770787172239387, 'colsample_bytree': 0.8100237087750082, 'reg_alpha': 6.914835872777325, 'reg_lambda': 2.3736995060475587e-05}. Best is trial 0 with value: 0.966714894722116.
[I 2026-04-28 15:14:02,516] Trial 2 finished with value: 0.9676599465124665 and parameters: {'n_estimators': 274, 'learning_rate': 0.0485858738163162, 'max_depth': 12, 'num_leaves': 100, 'min

---
# Model Training with Class Weights
---

---
## Setting Up the Parameters

This cell prepares the configuration for our LightGBM model. It uses a `try-except` block to intelligently load the `best_lgb_params` we discovered during the Optuna hyperparameter tuning. If, for some reason, the Optuna cell was not executed, it safely falls back to a set of robust default parameters so the code won't crash. Notice that `class_weight: 'balanced'` is explicitly defined here, which is the most critical setting for handling this dataset's class imbalance.

In [14]:
# Ensure parameters from Optuna are used, or fallback to default robust params
try:
    lgb_params = best_lgb_params
except NameError:
    print("Optuna params not found. Using fallback parameters...")
    lgb_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'boosting_type': 'gbdt',
        'class_weight': 'balanced', # Crucial for Imbalanced Dataset
        'random_state': RANDOM_STATE,
        'n_estimators': 600,
        'learning_rate': 0.05,
        'max_depth': 7,
        'num_leaves': 45
    }

fold_scores = []

---
## The K-Fold Training Loop

This is the core training engine. We iterate through the 5 stratified folds we created earlier. In each loop:
1. We train the model on 4/5 of the data and validate it on the remaining 1/5.
2. We use `early_stopping` to automatically halt training if the validation score stops improving, preventing the model from memorizing the data (overfitting).
3. The model outputs **probabilities** rather than hard classes using `predict_proba`. 
4. We save the validation predictions into our `oof_preds_lgb` array. Simultaneously, we ask the model to predict the unseen test dataset and add a fraction (`1/N_SPLITS`) of those probabilities into `test_preds_lgb`. By the end of the loop, the test predictions will be a perfectly averaged ensemble of all 5 folds.

In [15]:
# Iterate through the Stratified Folds defined in Step 3
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1}/{N_SPLITS} ---")
    
    # Split the data for current fold
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # Initialize the model with balanced class weights
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    
    # Train the model with early stopping to prevent overfitting
    model_lgb.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Generate probability predictions for validation fold and test set
    val_preds_proba = model_lgb.predict_proba(X_val_fold)
    test_preds_proba = model_lgb.predict_proba(X_test)
    
    # Store OOF probabilities and accumulate test predictions
    oof_preds_lgb[val_idx] = val_preds_proba
    test_preds_lgb += test_preds_proba / N_SPLITS
    
    # Calculate and store fold score
    val_preds_class = np.argmax(val_preds_proba, axis=1)
    fold_score = balanced_accuracy_score(y_val_fold, val_preds_class)
    fold_scores.append(fold_score)
    
    print(f"Fold {fold + 1} Balanced Accuracy: {fold_score:.5f}")


--- Training Fold 1/5 ---
Fold 1 Balanced Accuracy: 0.97033

--- Training Fold 2/5 ---
Fold 2 Balanced Accuracy: 0.97098

--- Training Fold 3/5 ---
Fold 3 Balanced Accuracy: 0.97234

--- Training Fold 4/5 ---
Fold 4 Balanced Accuracy: 0.97027

--- Training Fold 5/5 ---
Fold 5 Balanced Accuracy: 0.97068


---
## Final Out-Of-Fold Evaluation

After all 5 folds have finished training, we calculate the final **Out-Of-Fold (OOF) Score**. We take the aggregated probability predictions for the entire training dataset (`oof_preds_lgb`), convert them back into hard class predictions (0, 1, or 2) using `np.argmax`, and score them against the true labels. This Overall OOF score is the most reliable indicator of how well your model will perform on the Kaggle public leaderboard.

In [16]:
# Calculate Final OOF Score
final_oof_class = np.argmax(oof_preds_lgb, axis=1)
overall_oof_score = balanced_accuracy_score(y, final_oof_class)

print("\n--- Model Training Complete ---")
print(f"Average Fold Balanced Accuracy: {np.mean(fold_scores):.5f}")
print(f"Overall OOF Balanced Accuracy: {overall_oof_score:.5f}")


--- Model Training Complete ---
Average Fold Balanced Accuracy: 0.97092
Overall OOF Balanced Accuracy: 0.97092


---
# Ensemble / Model Blending (Soft Voting)
---

---
## Dynamic Weight Configuration

This cell intelligently prepares the rules for our ensemble (blending). It checks the sum of the `test_preds_xgb` and `test_preds_cat` arrays to verify if those models were actually trained (if the sum is greater than 0). Based on which models are available, it dynamically adjusts the blending weights. If you only trained LightGBM, it gives LightGBM 100% (1.0) of the voting power. If all three are trained, it splits the voting power among them, preventing the script from crashing or producing inaccurate zeros if a model was skipped.

In [17]:
is_xgb_trained = np.sum(test_preds_xgb) > 0
is_cat_trained = np.sum(test_preds_cat) > 0

weight_lgb = 1.0
weight_xgb = 0.0
weight_cat = 0.0

if is_xgb_trained and is_cat_trained:
    weight_lgb = 0.4
    weight_xgb = 0.3
    weight_cat = 0.3
elif is_xgb_trained:
    weight_lgb = 0.5
    weight_xgb = 0.5
elif is_cat_trained:
    weight_lgb = 0.5
    weight_cat = 0.5

print(f"Applied blending weights -> LGBM: {weight_lgb}, XGB: {weight_xgb}, Cat: {weight_cat}")

Applied blending weights -> LGBM: 1.0, XGB: 0.0, Cat: 0.0


---
## Soft Voting Execution

This is where the actual "Soft Voting" happens. Instead of taking the majority vote of the final predicted classes, soft voting takes the predicted *probabilities* from each model, multiplies them by their assigned weight, and averages them out. This method is generally superior because it takes into account how confident each model is about its prediction. Finally, we use `np.argmax` to look at the blended probabilities and pick the class (0, 1, or 2) that has the highest combined score.

In [18]:
# Calculate the weighted average of probabilities
final_test_preds_proba = (
    (test_preds_lgb * weight_lgb) + 
    (test_preds_xgb * weight_xgb) + 
    (test_preds_cat * weight_cat)
)

# Convert probabilities to final class predictions
final_test_preds_class = np.argmax(final_test_preds_proba, axis=1)

print(f"First 5 blended probabilities:\n{final_test_preds_proba[:5]}")
print(f"First 5 final predicted classes (encoded): {final_test_preds_class[:5]}")

First 5 blended probabilities:
[[9.99984594e-01 1.54064051e-05 7.07222354e-11]
 [8.06637529e-01 1.93333211e-01 2.92601244e-05]
 [9.99973565e-01 2.64338277e-05 1.51132790e-09]
 [9.95389885e-01 4.60913469e-03 9.80023474e-07]
 [9.99908454e-01 9.15442728e-05 2.00378959e-09]]
First 5 final predicted classes (encoded): [0 0 0 0 0]


---
# Final Prediction and Submission Formulation
---

---
## Reversing the Mapping and Building the DataFrame

In this step, we prepare our predictions to match Kaggle's required submission format. Because our models outputted numerical predictions (0, 1, or 2), we first create a `reverse_target_mapping` dictionary to translate those numbers back into their original text labels (`Low`, `Medium`, `High`). Finally, we construct a new Pandas DataFrame combining the `id` column from the test dataset with our newly translated `final_labels`.

In [19]:
# Define the reverse mapping to convert numeric classes back to original text labels
reverse_target_mapping = {0: 'Low', 1: 'Medium', 2: 'High'}

# Convert the blended numeric predictions back to string categories
final_labels = [reverse_target_mapping[pred] for pred in final_test_preds_class]

# Create the final submission dataframe using the 'id' column from the original test dataset
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': final_labels
})

---
## Exporting to CSV and Final Verification

This final cell exports our prepared DataFrame into a file named `submission.csv`. We explicitly use `index=False` so Pandas doesn't write an extra column of row numbers, which would cause an error when submitting to Kaggle. 

After saving, the code prints a preview of the first 10 rows and calculates the final percentage distribution of the predicted classes. This distribution check acts as a final sanity check to ensure our model didn't naively predict only the majority class (e.g., guessing 100% "Low"). You are now ready to submit!

In [20]:
# Save the dataframe to a CSV file without the index column
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)

print("\n--- Step 7 Complete ---")
print(f"Successfully saved predictions to '{submission_filename}'")
print("\nPreview of the submission file:")
print(submission_df.head(10))

print("\nPredicted Class Distribution (Percentage):")
print(submission_df['Irrigation_Need'].value_counts(normalize=True) * 100)

print("\nPipeline finished! You can now upload 'submission.csv' to Kaggle. Good luck!")


--- Step 7 Complete ---
Successfully saved predictions to 'submission.csv'

Preview of the submission file:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008            High
9  630009             Low

Predicted Class Distribution (Percentage):
Irrigation_Need
Low       59.164815
Medium    37.122963
High       3.712222
Name: proportion, dtype: float64

Pipeline finished! You can now upload 'submission.csv' to Kaggle. Good luck!
